<a href="https://colab.research.google.com/github/remyaP12/labcycle_3sem/blob/main/vehicle%2093_corr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SETTINGS YOU CAN TUNE
# ============================================================
SEQ_LEN = 30
TRAIN_RATIO = 0.8
MAX_EPOCHS = 5000
PATIENCE = 100
USE_SMOOTH_LOAD = True        # True = rolling mean target for engine load only
USE_CLUSTER_AS_FEATURE = True # True = 1 global LSTM with cluster feature; False = per-cluster LSTMs

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv('/content/OBD (1).csv')

features = {
    'rpm': 'Engine RPM(rpm)',
    'load': 'Engine Load(%)',
    'maf': 'Mass Air Flow Rate(g/s)',
    'throttle': 'Throttle Position(Manifold)(%)'
}

df_features = df[list(features.values())].copy()
print(f"Dataset: {df_features.shape} | Features: {list(features.values())}")

# Smooth engine load if requested (only affects the 'load' column)
if USE_SMOOTH_LOAD:
    df['Engine Load Smoothed'] = (
        df[features['load']].rolling(5, center=True).mean().bfill().ffill()
    )
    # Replace the original load column with the smoothed version
    df_features[features['load']] = df['Engine Load Smoothed']
    print("✅ Using smoothed Engine Load as target for load predictions.")
else:
    print("✅ Using raw Engine Load as target for load predictions.")

# ============================================================
# 2. SCALING (PAPER STYLE)
# ============================================================
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

df_features[[features['rpm'], features['maf']]] = scaler_minmax.fit_transform(
    df_features[[features['rpm'], features['maf']]]
)

df_features[[features['load']]] = scaler_minmax.fit_transform(
    df_features[[features['load']]]
)

df_features[[features['throttle']]] = scaler_robust.fit_transform(
    df_features[[features['throttle']]]
)

print("✅ Scaling complete.")

# ============================================================
# 3. K-MEANS CLUSTERING (K=2) ON ALL FOUR FEATURES
# ============================================================
X_cluster = df_features[list(features.values())].values
kmeans_final = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_cluster)
df_features['cluster'] = clusters

unique, counts = np.unique(clusters, return_counts=True)
print("Cluster sizes:", dict(zip(unique, counts)))

# ============================================================
# 4. MODEL BUILDERS (same as before)
# ============================================================
def build_lstm(input_shape):
    model = Sequential([
        LSTM(76, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(76),
        Dropout(0.5),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0002), loss='mse')
    return model

def build_gru(input_shape):
    model = Sequential([
        GRU(76, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        GRU(76),
        Dropout(0.5),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0002), loss='mse')
    return model

def build_rnn(input_shape):
    model = Sequential([
        SimpleRNN(76, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        SimpleRNN(76),
        Dropout(0.5),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0002), loss='mse')
    return model

def train_and_predict(build_fn, X_train, y_train, X_test):
    model = build_fn((X_train.shape[1], X_train.shape[2]))
    es = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True
    )
    model.fit(
        X_train, y_train,
        epochs=MAX_EPOCHS,
        batch_size=32,
        validation_split=0.1,
        callbacks=[es],
        verbose=0
    )
    y_pred = model.predict(X_test, verbose=0).ravel()
    return model, y_pred

# ============================================================
# 5. FUNCTION TO RUN EXPERIMENT FOR A GIVEN TARGET
# ============================================================
def run_for_target(target_name):
    print(f"\n{'='*60}")
    print(f"🔍 Processing target: {target_name.upper()}")
    print('='*60)

    # Find the column index of the target in the original feature list
    feature_cols = list(features.values())
    target_col = features[target_name]
    target_idx = feature_cols.index(target_col)  # 0: rpm, 1: load, 2: maf, 3: throttle

    # Build sequence data: base features + optionally cluster ID
    base_data = df_features[feature_cols].values
    if USE_CLUSTER_AS_FEATURE:
        seq_data = np.concatenate([base_data, clusters.reshape(-1, 1)], axis=1)
        # target index stays the same because cluster is appended at the end
        print("✅ Using cluster ID as extra feature.")
    else:
        seq_data = base_data
        print("✅ Will train per‑cluster models.")

    # Create sequences (predict the target column)
    X_all, y_all = create_sequences(seq_data, seq_length=SEQ_LEN, target_idx=target_idx)
    # Cluster labels for each sequence (shifted by SEQ_LEN)
    seq_clusters = clusters[SEQ_LEN:]

    # Global train/test split (time‑ordered)
    split = int(TRAIN_RATIO * len(X_all))
    X_train, X_test = X_all[:split], X_all[split:]
    y_train, y_test = y_all[:split], y_all[split:]
    clusters_train = seq_clusters[:split]
    clusters_test = seq_clusters[split:]

    # Train baseline models on global data
    print("⏳ Training LSTM...")
    _, y_pred_lstm = train_and_predict(build_lstm, X_train, y_train, X_test)

    print("⏳ Training GRU...")
    _, y_pred_gru = train_and_predict(build_gru, X_train, y_train, X_test)

    print("⏳ Training RNN...")
    _, y_pred_rnn = train_and_predict(build_rnn, X_train, y_train, X_test)

    # Hybrid model (proposed)
    if USE_CLUSTER_AS_FEATURE:
        # Global LSTM with cluster feature already trained (we reuse LSTM predictions)
        y_pred_hybrid = y_pred_lstm.copy()
        print("✅ Hybrid = global LSTM with cluster feature.")
    else:
        # Train one LSTM per cluster
        cluster_models = {}
        y_pred_hybrid = np.zeros_like(y_test)
        for c in np.unique(seq_clusters):
            idx_c = np.where(seq_clusters == c)[0]
            # Use only sequences belonging to this cluster
            X_c, y_c = X_all[idx_c], y_all[idx_c]
            # Train/test split within cluster
            split_c = int(TRAIN_RATIO * len(X_c))
            X_train_c, X_test_c = X_c[:split_c], X_c[split_c:]
            y_train_c, y_test_c = y_c[:split_c], y_c[split_c:]

            print(f"⏳ Training LSTM for cluster {c} (samples: {len(X_c)})...")
            model_c, _ = train_and_predict(build_lstm, X_train_c, y_train_c, X_test_c)
            cluster_models[c] = model_c

        # Predict on the global test set by routing each sample to its cluster model
        for i, (seq, c) in enumerate(zip(X_test, clusters_test)):
            model_c = cluster_models[c]
            y_pred_hybrid[i] = model_c.predict(seq[np.newaxis, ...], verbose=0)[0, 0]
        print("✅ Hybrid = per‑cluster LSTMs.")

    # Compute metrics
    mae_l, mse_l, rmse_l, r2_l = compute_metrics(y_test, y_pred_lstm)
    mae_g, mse_g, rmse_g, r2_g = compute_metrics(y_test, y_pred_gru)
    mae_r, mse_r, rmse_r, r2_r = compute_metrics(y_test, y_pred_rnn)
    mae_h, mse_h, rmse_h, r2_h = compute_metrics(y_test, y_pred_hybrid)

    return {
        'target': target_name,
        'LSTM': (mae_l, mse_l, rmse_l, r2_l),
        'GRU': (mae_g, mse_g, rmse_g, r2_g),
        'RNN': (mae_r, mse_r, rmse_r, r2_r),
        'Proposed': (mae_h, mse_h, rmse_h, r2_h)
    }

# Helper to create sequences
def create_sequences(data, seq_length=30, target_idx=1):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:i+seq_length])
        ys.append(data[i+seq_length, target_idx])
    return np.array(xs), np.array(ys)

# Helper to compute metrics
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

# ============================================================
# 6. RUN EXPERIMENTS FOR ALL TARGETS
# ============================================================
targets = ['throttle', 'rpm', 'maf', 'load']  # order as in prompt
results = []

for tgt in targets:
    res = run_for_target(tgt)
    results.append(res)

# ============================================================
# 7. BUILD AND PRINT SUMMARY TABLE (like the prompt)
# ============================================================
# Create a DataFrame with one row per target, columns for each model's metrics
rows = []
for res in results:
    row = {
        'Parameter': res['target'].capitalize(),
        'LSTM_MAE': res['LSTM'][0], 'LSTM_MSE': res['LSTM'][1],
        'LSTM_RMSE%': res['LSTM'][2]*100, 'LSTM_R2%': res['LSTM'][3]*100,
        'GRU_MAE': res['GRU'][0], 'GRU_MSE': res['GRU'][1],
        'GRU_RMSE%': res['GRU'][2]*100, 'GRU_R2%': res['GRU'][3]*100,
        'RNN_MAE': res['RNN'][0], 'RNN_MSE': res['RNN'][1],
        'RNN_RMSE%': res['RNN'][2]*100, 'RNN_R2%': res['RNN'][3]*100,
        'Proposed_MAE': res['Proposed'][0], 'Proposed_MSE': res['Proposed'][1],
        'Proposed_RMSE%': res['Proposed'][2]*100, 'Proposed_R2%': res['Proposed'][3]*100,
    }
    rows.append(row)

df_results = pd.DataFrame(rows).round(4)

# Display a compact version similar to the prompt
print("\n" + "="*80)
print("📊 Model performance comparison across all engine parameters")
print("="*80)

# For each parameter, print the proposed model's metrics as in the prompt
print("\n🔹 Proposed Model Performance (as described in prompt):")
proposed_summary = df_results[['Parameter', 'Proposed_MAE', 'Proposed_MSE', 'Proposed_RMSE%', 'Proposed_R2%']].copy()
proposed_summary.columns = ['Parameter', 'MAE', 'MSE', 'RMSE (%)', 'R² (%)']
print(proposed_summary.to_string(index=False))

# Also print the full comparison (optional)
print("\n🔹 Full comparison (all models):")
print(df_results.to_string(index=False))

# Optionally, save to CSV
# df_results.to_csv('model_comparison_all_parameters.csv', index=False)

Dataset: (1356, 4) | Features: ['Engine RPM(rpm)', 'Engine Load(%)', 'Mass Air Flow Rate(g/s)', 'Throttle Position(Manifold)(%)']
✅ Using smoothed Engine Load as target for load predictions.
✅ Scaling complete.
Cluster sizes: {np.int32(0): np.int64(471), np.int32(1): np.int64(885)}

🔍 Processing target: THROTTLE
✅ Using cluster ID as extra feature.
⏳ Training LSTM...
⏳ Training GRU...
⏳ Training RNN...
✅ Hybrid = global LSTM with cluster feature.

🔍 Processing target: RPM
✅ Using cluster ID as extra feature.
⏳ Training LSTM...
⏳ Training GRU...
⏳ Training RNN...
✅ Hybrid = global LSTM with cluster feature.

🔍 Processing target: MAF
✅ Using cluster ID as extra feature.
⏳ Training LSTM...
⏳ Training GRU...
⏳ Training RNN...
✅ Hybrid = global LSTM with cluster feature.

🔍 Processing target: LOAD
✅ Using cluster ID as extra feature.
⏳ Training LSTM...
⏳ Training GRU...
⏳ Training RNN...
✅ Hybrid = global LSTM with cluster feature.

📊 Model performance comparison across all engine parameter